In [1]:
import os
import glob
import numpy as np

In [2]:
# Parámetros de configuración
current_dataset_dir = 'data'      # Carpeta del dataset original
desired_total = 620000            # Total deseado después de la ampliación
num_points = 5000                 # Cada espectro tiene 5000 puntos

# Rango de redshift para seleccionar nuevos espectros
z_min = 0.0
z_max = 8.0

# Cargar los redshifts existentes del dataset actual
current_redshift_path = os.path.join(current_dataset_dir, 'spectra_data_450k_redshift_mmap.dat')
current_total = 460000  # Número actual de espectros
current_redshifts = np.memmap(current_redshift_path, dtype='float32', mode='r', shape=(current_total,))
existing_redshifts = set(current_redshifts.tolist())

# Cargar el nuevo dataset (3M espectros) desde archivos .dat
# Asumimos que los archivos nuevos se han guardado con np.memmap y tienen las siguientes rutas:
new_flux_path = os.path.join(current_dataset_dir, 'spectra_data_complete_flux_mmap.dat')
new_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_complete_wavelength_mmap.dat')
new_redshift_path = os.path.join(current_dataset_dir, 'spectra_data_complete_redshift_mmap.dat')

new_total_available = 3372890  # Total de espectros en el nuevo dataset

# Cargar los memmaps del nuevo dataset en modo lectura
new_flux = np.memmap(new_flux_path, dtype='float32', mode='r', shape=(new_total_available, num_points))
new_wavelength = np.memmap(new_wavelength_path, dtype='float32', mode='r', shape=(new_total_available, num_points))
new_redshifts = np.memmap(new_redshift_path, dtype='float32', mode='r', shape=(new_total_available,))

new_flux_list = []
new_wavelength_list = []
new_redshift_list = []

print("Filtrando nuevos espectros del dataset de 3M...")
# Recorrer el nuevo dataset
for i in range(new_total_available):
    r = new_redshifts[i]
    # Filtrar por rango de redshift
    if r < z_min or r > z_max:
        continue
    # Evitar duplicados: se comprueba que el redshift no exista ya en el dataset original
    if r in existing_redshifts:
        continue
    # Si cumple ambas condiciones, se añade la información
    new_flux_list.append(new_flux[i, :].copy())         # .copy() para obtener un array independiente
    new_wavelength_list.append(new_wavelength[i, :].copy())
    new_redshift_list.append(r)
    existing_redshifts.add(r)  # Agregar para evitar duplicados posteriores
    
    # Si se alcanza el total deseado, se finaliza el filtrado
    if current_total + len(new_redshift_list) >= desired_total:
        break

print(f"Se han encontrado {len(new_redshift_list)} nuevos espectros en el rango de redshift [{z_min}, {z_max}].")

# Crear el nuevo dataset ampliado
new_total = current_total + len(new_redshift_list)
print(f"Dataset ampliado: {new_total} espectros.")

# Rutas para los nuevos archivos memmap actualizados
new_flux_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_flux_mmap.dat')
new_wavelength_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_wavelength_mmap.dat')
new_redshift_mmap_path = os.path.join(current_dataset_dir, 'spectra_data_500k_redshift_mmap.dat')

# Preasignar los memmaps para el dataset ampliado
flux_mmap_new = np.memmap(new_flux_mmap_path, dtype='float32', mode='w+', shape=(new_total, num_points))
wavelength_mmap_new = np.memmap(new_wavelength_mmap_path, dtype='float32', mode='w+', shape=(new_total, num_points))
redshift_mmap_new = np.memmap(new_redshift_mmap_path, dtype='float32', mode='w+', shape=(new_total,))

# Copiar los datos antiguos del dataset actual
current_flux_path = os.path.join(current_dataset_dir, 'spectra_data_450k_flux_mmap.dat')
current_wavelength_path = os.path.join(current_dataset_dir, 'spectra_data_450k_wavelength_mmap.dat')

flux_mmap_old = np.memmap(current_flux_path, dtype='float32', mode='r', shape=(current_total, num_points))
wavelength_mmap_old = np.memmap(current_wavelength_path, dtype='float32', mode='r', shape=(current_total, num_points))

# Copiar los datos del dataset original en los nuevos memmaps
flux_mmap_new[:current_total, :] = flux_mmap_old[:]
wavelength_mmap_new[:current_total, :] = wavelength_mmap_old[:]
redshift_mmap_new[:current_total] = current_redshifts[:]

# Añadir los nuevos espectros filtrados
for i, (flux_array, wave_array, r) in enumerate(zip(new_flux_list, new_wavelength_list, new_redshift_list)):
    idx = current_total + i
    flux_mmap_new[idx, :] = flux_array
    wavelength_mmap_new[idx, :] = wave_array
    redshift_mmap_new[idx] = r

# Asegurarse de que los cambios se guarden en disco
flux_mmap_new.flush()
wavelength_mmap_new.flush()
redshift_mmap_new.flush()

print("Se ha actualizado el dataset ampliado y se han guardado los nuevos archivos memmap.")

Filtrando nuevos espectros del dataset de 3M...
Se han encontrado 160000 nuevos espectros en el rango de redshift [0.0, 8.0].
Dataset ampliado: 620000 espectros.
Se ha actualizado el dataset ampliado y se han guardado los nuevos archivos memmap.


In [2]:
import os
import numpy as np

original_dir = 'data'
num_points = 5000

flux_orig_path = os.path.join(original_dir, 'spectra_data_trainingbig_flux.dat')
wavelength_orig_path = os.path.join(original_dir, 'spectra_data_trainingbig_wavelength.dat')
redshift_orig_path = os.path.join(original_dir, 'spectra_data_trainingbig_redshift.dat')
# Total de espectros en el archivo original ampliado
total_orig = 620000

# Rango de redshift a trasladar
z_min, z_max = 4.0, 8.0

# Cargar los memmaps originales
flux_orig = np.memmap(flux_orig_path, dtype='float32', mode='r', shape=(total_orig, num_points))
wavelength_orig = np.memmap(wavelength_orig_path, dtype='float32', mode='r', shape=(total_orig, num_points))
redshift_orig = np.memmap(redshift_orig_path, dtype='float32', mode='r', shape=(total_orig,))

# Crear la máscara para el rango deseado
mask = (redshift_orig >= z_min) & (redshift_orig <= z_max)
indices_to_move = np.where(mask)[0]
indices_to_keep = np.where(~mask)[0]

print("Número de espectros a trasladar:", len(indices_to_move))
print("Número de espectros restantes:", len(indices_to_keep))

# Preasignar nuevos archivos memmap para el subconjunto filtrado
subset_total = len(indices_to_move)
new_flux_subset_path = os.path.join(original_dir, 'spectra_data_subset_flux_mmap.dat')
new_wavelength_subset_path = os.path.join(original_dir, 'spectra_data_subset_wavelength_mmap.dat')
new_redshift_subset_path = os.path.join(original_dir, 'spectra_data_subset_redshift_mmap.dat')

flux_subset = np.memmap(new_flux_subset_path, dtype='float32', mode='w+', shape=(subset_total, num_points))
wavelength_subset = np.memmap(new_wavelength_subset_path, dtype='float32', mode='w+', shape=(subset_total, num_points))
redshift_subset = np.memmap(new_redshift_subset_path, dtype='float32', mode='w+', shape=(subset_total,))

# Copiar las filas filtradas a los nuevos archivos
for new_idx, orig_idx in enumerate(indices_to_move):
    flux_subset[new_idx, :] = flux_orig[orig_idx, :]
    wavelength_subset[new_idx, :] = wavelength_orig[orig_idx, :]
    redshift_subset[new_idx] = redshift_orig[orig_idx]

flux_subset.flush()
wavelength_subset.flush()
redshift_subset.flush()
print("Se han creado los nuevos archivos con el subconjunto de espectros.")

# Crear nuevos archivos para el dataset original sin el subconjunto trasladado
remaining_total = len(indices_to_keep)
new_flux_remaining_path = os.path.join(original_dir, 'spectra_data_remaining_flux_mmap.dat')
new_wavelength_remaining_path = os.path.join(original_dir, 'spectra_data_remaining_wavelength_mmap.dat')
new_redshift_remaining_path = os.path.join(original_dir, 'spectra_data_remaining_redshift_mmap.dat')

flux_remaining = np.memmap(new_flux_remaining_path, dtype='float32', mode='w+', shape=(remaining_total, num_points))
wavelength_remaining = np.memmap(new_wavelength_remaining_path, dtype='float32', mode='w+', shape=(remaining_total, num_points))
redshift_remaining = np.memmap(new_redshift_remaining_path, dtype='float32', mode='w+', shape=(remaining_total,))

for new_idx, orig_idx in enumerate(indices_to_keep):
    flux_remaining[new_idx, :] = flux_orig[orig_idx, :]
    wavelength_remaining[new_idx, :] = wavelength_orig[orig_idx, :]
    redshift_remaining[new_idx] = redshift_orig[orig_idx]

flux_remaining.flush()
wavelength_remaining.flush()
redshift_remaining.flush()
print("Se han creado nuevos archivos para el dataset sin los espectros trasladados.")

Número de espectros a trasladar: 4528
Número de espectros restantes: 615472
Se han creado los nuevos archivos con el subconjunto de espectros.
Se han creado nuevos archivos para el dataset sin los espectros trasladados.


In [1]:
import os
import numpy as np

# Parámetros
num_points = 5000
total_remaining = 607472 # Número de espectros en el archivo remaining

# Rutas de los archivos "remaining"
data_dir = "data"
remaining_flux_path = os.path.join(data_dir, "spectra_data_remaining_flux_mmap.dat")
remaining_wavelength_path = os.path.join(data_dir, "spectra_data_remaining_wavelength_mmap.dat")
remaining_redshift_path = os.path.join(data_dir, "spectra_data_remaining_redshift_mmap.dat")

# Cargar los memmaps originales en modo lectura
flux_remaining = np.memmap(remaining_flux_path, dtype="float32", mode="r", shape=(total_remaining, num_points))
wavelength_remaining = np.memmap(remaining_wavelength_path, dtype="float32", mode="r", shape=(total_remaining, num_points))
redshift_remaining = np.memmap(remaining_redshift_path, dtype="float32", mode="r", shape=(total_remaining,))

# Definir el rango de redshift del que se eliminarán datos
delete_z_min = 0.0   # Valor inferior del rango
delete_z_max = 93.4   # Valor superior del rango

# Crear una máscara para los espectros cuyo redshift está dentro del rango
mask_in_range = (redshift_remaining >= delete_z_min) & (redshift_remaining <= delete_z_max)
indices_in_range = np.where(mask_in_range)[0]

print("Número total de espectros en el rango [{}, {}]: {}".format(delete_z_min, delete_z_max, len(indices_in_range)))

# Número de espectros que queremos eliminar
num_to_remove = 15472 + 4528 - 8000
# 0.84 para el train set

# Seleccionar exactamente 200 índices para eliminar (o todos si hay menos de 200)
if len(indices_in_range) < num_to_remove:
    print("No hay suficientes espectros en el rango; se eliminarán {} espectros.".format(len(indices_in_range)))
    indices_to_remove = indices_in_range
    num_to_remove = len(indices_in_range)
else:
    indices_to_remove = indices_in_range[:num_to_remove]  # También podrías elegirlos de forma aleatoria

# Crear una máscara de booleanos para conservar las filas que NO se eliminarán
mask_keep = np.ones(total_remaining, dtype=bool)
mask_keep[indices_to_remove] = False

# Calcular el nuevo total final
new_total_final = int(np.sum(mask_keep))
print("Número de espectros finales después de eliminar {}: {}".format(num_to_remove, new_total_final))

# Rutas para los nuevos archivos memmap finales
new_flux_final_path = os.path.join(data_dir, "spectra_data_final_flux_mmap.dat")
new_wavelength_final_path = os.path.join(data_dir, "spectra_data_final_wavelength_mmap.dat")
new_redshift_final_path = os.path.join(data_dir, "spectra_data_final_redshift_mmap.dat")

# Preasignar los nuevos archivos memmap con la nueva forma
flux_final = np.memmap(new_flux_final_path, dtype="float32", mode="w+", shape=(new_total_final, num_points))
wavelength_final = np.memmap(new_wavelength_final_path, dtype="float32", mode="w+", shape=(new_total_final, num_points))
redshift_final = np.memmap(new_redshift_final_path, dtype="float32", mode="w+", shape=(new_total_final,))

# Copiar las filas que se mantienen (aquellas para las cuales mask_keep es True)
keep_indices = np.where(mask_keep)[0]
for new_idx, orig_idx in enumerate(keep_indices):
    flux_final[new_idx, :] = flux_remaining[orig_idx, :]
    wavelength_final[new_idx, :] = wavelength_remaining[orig_idx, :]
    redshift_final[new_idx] = redshift_remaining[orig_idx]

# Asegurarse de guardar los cambios en disco
flux_final.flush()
wavelength_final.flush()
redshift_final.flush()

print("Se han generado los nuevos archivos sin los 200 espectros eliminados.")

Número total de espectros en el rango [0.0, 93.4]: 607314
Número de espectros finales después de eliminar 12000: 595472
Se han generado los nuevos archivos sin los 200 espectros eliminados.
